In [1]:
import pulp

# **Data**

In [2]:
def process_job_data(job_data):
    """
    Transforms raw job sequence data into the mathematical sets required for the
    Job Shop Scheduling LP model (Nodes, Solid Arcs, Broken Arcs).

    Args:
        job_data (list): List of jobs, where each job is a list of (machine_id, duration).

    Returns:
        tuple: (nodes, solid_arcs, broken_arcs, processing_times, big_m)
    """
    num_jobs = len(job_data)
    # Infer number of machines (max index found + 1)
    num_machines = max(max(op[0] for op in job) for job in job_data) + 1

    # --- Initialize Sets ---
    nodes = []              # Set N: (machine, job)
    solid_arcs = []         # Set A: Conjunctive arcs ((prev_m, j), (curr_m, j))
    broken_arcs = []        # Set B: Disjunctive triplets (machine, job_j, job_k)
    processing_times = {}   # Parameter p_{mj}
    
    # Helper to track operations per machine for building Broken Arcs
    machine_ops_map = {m: [] for m in range(num_machines)}

    # --- Build Nodes (N) and Solid Arcs (A) ---
    for j, job_seq in enumerate(job_data):
        prev_node = None
        for (m, p) in job_seq:
            curr_node = (m, j)
            
            # Add to Nodes and record time
            nodes.append(curr_node)
            processing_times[curr_node] = p
            
            # Track for Broken Arcs later
            machine_ops_map[m].append(j)
            
            # Add Solid Arc (Precedence: Previous -> Current)
            if prev_node:
                solid_arcs.append((prev_node, curr_node))
            prev_node = curr_node

    # --- Build Broken Arcs (B) ---
    # Create pairs for all jobs sharing the same machine
    for m in range(num_machines):
        jobs = machine_ops_map[m]
        for i in range(len(jobs)):
            for k in range(i + 1, len(jobs)):
                job_a = jobs[i]
                job_b = jobs[k]
                # Store as (Machine, Job1, Job2)
                # We enforce an ordering (a < b) to ensure unique variable creation
                if job_a < job_b:
                    broken_arcs.append((m, job_a, job_b))
                else:
                    broken_arcs.append((m, job_b, job_a))

    return nodes, solid_arcs, broken_arcs, processing_times

# **JSP LP relax model**

In [5]:
def create_jsp_relaxed_model(nodes, solid_arcs, broken_arcs, processing_times):
    """
    Creates the Job Shop Scheduling LP Relaxation model using pre-calculated sets.
    Based on the formulation in 'JSP LP relax model.pdf' (Section 3.1).

    Args:
        nodes (list): List of all operation nodes (machine, job).
        solid_arcs (list): List of conjunctive arcs ((m1,j), (m2,j)).
        broken_arcs (list): List of disjunctive triplets (machine, job_j, job_k).
        processing_times (dict): Map of (machine, job) -> duration.
        big_m (float): Large constant for linearization.
    """
    
    # --- 1. Initialize Model ---
    # Minimization problem for Makespan
    mdl = pulp.LpProblem("JSP_Relaxed_MIP", pulp.LpMinimize)

    # --- 2. Create Variables ---
    
    # y_{mj}: Start time of job j on machine m (Continuous, >= 0)
    y = {}
    for node in nodes:
        y[node] = pulp.LpVariable(f"Start_{node}", 0, None, pulp.LpContinuous)

    # x_{mjk}: Precedence binary variables for disjunctive arcs
    # x = 1 if j precedes k on machine m
    x = {}
    for (m, j, k) in broken_arcs:
        x[(m, j, k)] = pulp.LpVariable(f"X_m{m}_j{j}_j{k}", 0, 1, pulp.LpContinuous)

    # Cmax: Makespan variable
    Cmax = pulp.LpVariable("Cmax", 0, None, pulp.LpContinuous)

    # --- 3. Objective Function ---
    # [cite_start]Minimize Cmax [cite: 107]
    mdl += Cmax, "Minimize_Makespan"

    # --- 4. Constraints ---

    # (A) Conjunctive Constraints (Solid Arcs - Equation 109)
    # y_{hj} >= y_{mj} + p_{mj}
    # Enforces the technical sequence of operations for each job.
    for (prev_node, curr_node) in solid_arcs:
        p_mj = processing_times[prev_node]
        mdl += y[curr_node] >= y[prev_node] + p_mj, f"Seq_{prev_node}_to_{curr_node}"

    # (B) Disjunctive Constraints (Broken Arcs - Equation 111)
    # Enforces that two jobs cannot use the same machine simultaneously.
    big_m = 10**6
    for (m, j, i) in broken_arcs:
        node_j = (m, j)
        node_i = (m, i)
        p_j = processing_times[node_j]
        p_i = processing_times[node_i]
        
        # If j precedes i (x=1): y_i >= y_j + p_j
        mdl += y[node_i] >= y[node_j] + p_j - big_m * (1 - x[(m, j, i)]), f"Disj_M{m}_{j}_before_{i}"
        
        # If k precedes j (x=0): y_j >= y_k + p_k
        mdl += y[node_j] >= y[node_i] + p_i - big_m * x[(m, j, i)], f"Disj_M{m}_{i}_before_{j}"

    # (C) Makespan Definition (Equation 110)
    # Cmax >= y_{mj} + p_{mj} for all operations
    for node in nodes:
        p_mj = processing_times[node]
        mdl += Cmax >= y[node] + p_mj, f"Cmax_bound_{node}"

    return mdl, y, x, Cmax

# **Execution**

In [6]:
# 1. Raw Data (ft06)
ft06_data = [
    [(2, 1), (0, 3), (1, 6), (3, 7), (5, 3), (4, 6)],
    [(1, 8), (2, 5), (4, 10), (5, 10), (0, 10), (3, 4)],
    [(2, 5), (3, 4), (5, 8), (0, 9), (1, 1), (4, 7)],
    [(1, 5), (0, 5), (2, 5), (3, 3), (4, 8), (5, 9)],
    [(2, 9), (1, 3), (4, 5), (5, 4), (0, 3), (3, 1)],
    [(1, 3), (3, 3), (5, 9), (0, 10), (4, 4), (2, 1)],
]

print("--- 1. Processing Data ---")
# Transform raw data into mathematical sets
nodes, solid, broken, p_times= process_job_data(ft06_data)

print(f"   Nodes: {len(nodes)}")
print(f"   Solid Arcs: {len(solid)}")
print(f"   Broken Arcs: {len(broken)}")

print("\n--- 2. Building & Solving Model ---")
model, y_vars, x_vars, cmax = create_jsp_relaxed_model(nodes, solid, broken, p_times)

# Solve
solver = pulp.PULP_CBC_CMD(msg=False)
model.solve(solver)

print(f"   Status: {pulp.LpStatus[model.status]}")
print(f"   Makespan: {pulp.value(cmax)}")

# Print Schedule
print("\n--- 3. Schedule ---")
print(f"{'Job':<5} {'Machine':<10} {'Start':<10} {'Finish':<10}")
print("-" * 40)

results = []
for (m, j) in nodes:
    start = pulp.value(y_vars[(m, j)])
    dur = p_times[(m, j)]
    results.append((j, m, start, start+dur))
    
for res in sorted(results, key=lambda x: (x[0], x[2])):
    print(f"{res[0]:<5} {res[1]:<10} {res[2]:<10g} {res[3]:<10g}")

--- 1. Processing Data ---
   Nodes: 36
   Solid Arcs: 30
   Broken Arcs: 90

--- 2. Building & Solving Model ---
   Status: Optimal
   Makespan: 47.0

--- 3. Schedule ---
Job   Machine    Start      Finish    
----------------------------------------
0     2          0          1         
0     0          1          4         
0     1          4          10        
0     3          10         17        
0     5          17         20        
0     4          20         26        
1     1          0          8         
1     2          8          13        
1     4          13         23        
1     5          23         33        
1     0          33         43        
1     3          43         47        
2     2          0          5         
2     3          5          9         
2     5          9          17        
2     0          17         26        
2     1          26         27        
2     4          27         34        
3     1          0          5         
3     0